# Text → STL: end-to-end pipeline
Type a plain-English description of a **pipe gasket, washer/spacer, or L-bracket** → get a checked parameter table → confirm → download an STL.

**How to run:** Runtime ▸ Change runtime type ▸ **T4 GPU** → run the cells top to bottom (Runtime ▸ Run all). When asked, upload `24679_Project1_Data.xlsx`. The last-but-one cell starts the app and shows it right in the notebook (plain HTML page + a small Flask server, no Gradio).

| Your pipeline diagram box | Where it lives |
|---|---|
| Input text description → GUI text box | GUI cell (§10) |
| Preprocess to numeric decimal form | §2 `normalize_text` |
| Trained-from-scratch model → part type | §6 (PyTorch classifier, random init) |
| Fine-tuned model → parameters | §7 (LoRA on Qwen2.5) |
| JSON with standard schema for all parts | §3 `SCHEMA`, `params_to_json` |
| Validation (negatives, missing, range) + off-the-shelf model verifies? | §4 rules (always on) + optional LLM verifier (`USE_LLM_VERIFIER`) |
| Table for the user to confirm/edit → confirm button | GUI cell (§10) |
| JSON → CAD script → CAD file → 3D preview, download, start over | §5 CAD + GUI cell |
| User edits loop back | changing the part-type dropdown re-runs the extractor; every confirmed edit is logged to `work/feedback.jsonl` |

The deterministic parts (preprocessing, validation, CAD) run and self-test **before** the slow training cells, so a problem there shows up in seconds.

In [ ]:
# Runtime > Change runtime type > T4 GPU (needed for the fine-tuning cell). Then run this cell once.
# If Colab asks you to restart the runtime after installing, do that and continue from the next cell.
!pip -q install cadquery trimesh flask "transformers>=4.44" peft accelerate openpyxl scikit-learn
# Colab ships an old torchao (0.10) that newer peft refuses to import; we don't use it, so remove it.
!pip -q uninstall -y torchao

## 0. Setup & settings
Everything you might want to tune is at the top of this cell.

In [ ]:
import os, re, json, math, random, time, zipfile, copy
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ CONFIG
SEED = 42
DATA_PATH = "24679_Project1_Data.xlsx"      # if this file isn't found, Colab will ask you to upload it
WORKDIR = Path("work"); WORKDIR.mkdir(exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # fine-tuned model. Use "Qwen/Qwen2.5-0.5B-Instruct" for a faster/lighter run
LORA_EPOCHS = 8
LORA_LR = 2e-4
BATCH_SIZE = 8

TARGET_MANUAL_SAMPLES = 500                 # assignment goal, only used for a progress line
EXCLUDE_AUDIT_FLAGGED = False               # True = drop rows whose label numbers don't appear in their text
GROUP_SPLIT = True                          # True = a given pipe size / screw size never appears in both train and test
TEST_FRAC, VAL_FRAC = 0.15, 0.15

NUM_TOL = 0.0075                            # inches: tolerance for "value appears in text" and for eval correctness
SNAP_TO_STANDARD = True                     # snap values within NUM_TOL of a known catalog size to that size
USE_LLM_VERIFIER = False                    # optional off-the-shelf LLM check (weak with small models, see Stage 3)

random.seed(SEED); np.random.seed(SEED)

def show_df(df):
    """pretty table in Colab/Jupyter, plain text elsewhere"""
    try: display(df)
    except NameError: print(df.to_string())

## 1. Load and clean the spreadsheet
All sheets are combined. The `L_brackets` sheet's text column is headed `n`; it is renamed to `text`. Rows without text stay available for the catalog tables but can't be used to train the language models.

In [ ]:
# ------------------------------------------------------------------ LOAD + CLEAN THE SPREADSHEET
if not Path(DATA_PATH).exists():
    try:
        from google.colab import files            # only exists inside Colab
        print("Upload your project spreadsheet (.xlsx)...")
        _up = files.upload()
        DATA_PATH = next(iter(_up))
    except ImportError:
        raise FileNotFoundError(f"Could not find {DATA_PATH}. Put it next to this notebook.")

xl = pd.ExcelFile(DATA_PATH)
frames = []
for sheet in xl.sheet_names:
    d = xl.parse(sheet)
    if "text" not in d.columns and "n" in d.columns:   # the L_brackets sheet's text column is headed "n"
        d = d.rename(columns={"n": "text"})
    d["sheet"] = sheet
    d["sheet_row"] = np.arange(len(d)) + 2              # Excel row number (header = row 1)
    frames.append(d)
raw_all = pd.concat(frames, ignore_index=True)
raw_all["text"] = raw_all["text"].astype("string").str.strip()

has_text = raw_all["text"].notna() & (raw_all["text"] != "")
summary = raw_all.assign(has_text=has_text).groupby("sheet").agg(rows=("has_text", "size"), with_text=("has_text", "sum"))
summary["missing_text"] = summary["rows"] - summary["with_text"]
print(summary, "\n")

df_all = raw_all[has_text].copy().reset_index(drop=True)
print(f"Rows with real text (used for ML): {len(df_all)}  |  goal: {TARGET_MANUAL_SAMPLES}  |  still needed: {max(0, TARGET_MANUAL_SAMPLES - len(df_all))}")
print("Rows without text are kept for lookup tables / CAD standards, but not used to train the language models.")

## 2. Preprocessing — everything to decimal numbers
Fractions (`1 1/16`), mixed numbers, word fractions (`a sixteenth`, `three and a half`), number words (`eight bolt holes`, `a dozen`), leading-dot decimals (`.0625`), glued units (`4.62OD`), and screw-size phrasing (`triple aught`, `quarter twenty`, `#8`, `metric 3.9`).  
The **same function** runs on training text and on what a user types, so the models never see two different formats.

In [ ]:
# ------------------------------------------------------------------ PREPROCESSING: everything -> decimal numbers
# Pipeline box: "Data is preprocessed to make all dimensions and numbers numeric decimal form".
# The SAME function is used for training text and for live user input (no train/serve skew).

NUM_WORDS = {"zero":0,"one":1,"two":2,"three":3,"four":4,"five":5,"six":6,"seven":7,"eight":8,"nine":9,"ten":10,
             "eleven":11,"twelve":12,"thirteen":13,"fourteen":14,"fifteen":15,"sixteen":16,"seventeen":17,
             "eighteen":18,"nineteen":19,"twenty":20,"thirty":30,"forty":40,"fifty":50}
_UNITS = "one|two|three|four|five|six|seven|eight|nine"
_WORDNUM = "|".join(sorted(NUM_WORDS, key=len, reverse=True))
WORD_FRACS = {"sixteenth": 1/16, "eighth": 1/8, "quarter": 1/4, "thirty-second": 1/32, "sixty-fourth": 1/64}

def dec(x, places=4):
    """0.0625 -> '0.0625', 3.5 -> '3.5', 4.0 -> '4'  (max 4 decimals, no trailing zeros)"""
    return f"{x:.{places}f}".rstrip("0").rstrip(".")

def _word_or_digit(s):
    s = s.lower()
    return float(NUM_WORDS[s]) if s in NUM_WORDS else float(s)

_AUGHT   = [(re.compile(r"\btriple[- ]aught\b", re.I), "No. 000"),
            (re.compile(r"\bdouble[- ]aught\b", re.I), "No. 00"),
            (re.compile(r"\bsingle[- ]aught\b", re.I), "No. 0")]
_QTR20   = re.compile(r"\bquarter[- ]twenty\b", re.I)
_FRACWORD = {"half": 0.5, "quarter": 0.25, "eighth": 0.125, "sixteenth": 0.0625, "three quarters": 0.75, "three-quarters": 0.75}
_AND_FRAC = re.compile(rf"\b(an|{_WORDNUM}|\d+(?:\.\d+)?)(?:\s+inch(?:es)?)?\s+and\s+(?:(?:a|an)\s+(half|quarter|eighth|sixteenth)|(three[- ]quarters))\b(?:\s+of\s+an)?(?:\s+inch(?:es)?)?", re.I)
_DOZEN   = re.compile(r"\b(?:a\s+)?dozen\b", re.I)
_HALF_IN = re.compile(r"\b(?:a\s+)?half(?:[- ]an?)?[- ]inch(?:es)?\b", re.I)
_AN_INCH = re.compile(r"\ban\s+inch\b", re.I)
_WORDFRAC= re.compile(r"\b(?:an?\s+)?(sixteenth|eighth|quarter|thirty[- ]second|sixty[- ]fourth)\b(?:\s+of\s+an)?(?:\s+inch(?:es)?)?", re.I)
_COUNT_CTX = r'(?=\s*(?:inch|inches|in\b|"|bolts?\b|holes?\b|mounting\b|mount\b|rivets?\b|points?\b|hardware\b|screws?\b))'
_TENS_UNITS = re.compile(rf"\b(twenty|thirty|forty|fifty)[- ]({_UNITS})\b{_COUNT_CTX}", re.I)
_WORDNUM_CTX= re.compile(rf"\b({_WORDNUM})\b{_COUNT_CTX}", re.I)
_NO_SIZE = re.compile(r"(?<!\w)(?:no\.?|number|#)\s*(\d+)\b", re.I)
_SIZE_N  = re.compile(r"\bsize\s+(\d+)\b(?=\s+(?:screw|bolt|hardware))", re.I)
_METRIC  = re.compile(r"\bmetric\s+(\d+(?:\.\d+)?)\b", re.I)
_MIXED   = re.compile(r"(?<![\d.])(\d+)\s+(\d+)/(\d+)(?!\d)")
_FRAC    = re.compile(r"(?<![\d.])(\d+)/(\d+)(?!\d)")
_LEADDOT = re.compile(r"(?<!\d)\.(\d+)")
_GLUE    = re.compile(r"(\d)([A-Za-z])")

def normalize_text(text: str) -> str:
    t = str(text)
    for pat, rep in _AUGHT: t = pat.sub(rep, t)
    t = _QTR20.sub("0.25", t)
    t = _DOZEN.sub("12", t)
    t = _AND_FRAC.sub(lambda m: dec((1.0 if m.group(1).lower() == "an" else _word_or_digit(m.group(1)))
                                    + (0.75 if m.group(3) else _FRACWORD[m.group(2).lower()])) + " inch", t)
    t = _HALF_IN.sub("0.5 inch", t)
    t = _AN_INCH.sub("1 inch", t)
    t = _WORDFRAC.sub(lambda m: dec(WORD_FRACS[re.sub(r"\s", "-", m.group(1).lower())]) + " inch", t)
    t = _TENS_UNITS.sub(lambda m: str(NUM_WORDS[m.group(1).lower()] + NUM_WORDS[m.group(2).lower()]), t)
    t = _WORDNUM_CTX.sub(lambda m: str(NUM_WORDS[m.group(1).lower()]), t)
    t = _NO_SIZE.sub(lambda m: f"No. {m.group(1)}", t)
    t = _SIZE_N.sub(lambda m: f"No. {m.group(1)}", t)
    t = _METRIC.sub(lambda m: f"M{m.group(1)}", t)
    t = _MIXED.sub(lambda m: dec(int(m.group(1)) + int(m.group(2)) / int(m.group(3))), t)
    t = _FRAC.sub(lambda m: dec(int(m.group(1)) / int(m.group(2))), t)
    t = _LEADDOT.sub(r"0.\1", t)
    t = _GLUE.sub(r"\1 \2", t)
    return re.sub(r"\s+", " ", t).strip()

def numbers_in(text_norm: str):
    """All standalone numbers in already-normalized text (used for the 'is this value in the text?' check)."""
    return [float(x) for x in re.findall(r"(?<![\w.])\d+(?:\.\d+)?", text_norm)]

def tol_for(v: float) -> float:
    """How close counts as 'the same number'. Absolute 0.0075 in for normal sizes, but much tighter for tiny values
    (a 0.012" washer must not be confused with a 0.015" one)."""
    return min(NUM_TOL, max(0.0006, 0.05 * abs(v)))

# ---- quick self-tests (they run every time so a broken edit is caught immediately)
_tests = {
    'make a pipe gasket 1 1/16" ID and 3 7/8" OD. 1/16" thick': ['1.0625"', '3.875"', '0.0625"'],
    "They should be a sixteenth inch thick": ["0.0625 inch thick"],
    "3 and a half inch pipe with a seven and a half inch flange": ["3.5 inch pipe", "7.5 inch flange"],
    "fastened at four mount points, eight bolt holes": ["4 mount points", "8 bolt holes"],
    "4.62OD and 1.66ID, 3.75od, 3.5in": ["4.62 OD", "1.66 ID", "3.75 od", "3.5 in"],
    "washer for a triple aught screw": ["No. 000 screw"],
    "a quarter twenty screw": ["0.25 screw"],
    "for a no1 screw": ["No. 1 screw"],
    "spacer for a #8 screw": ["No. 8 screw"],
    "washer for a size ten screw": ["No. 10 screw"],
    "for a metric 3.9 bolt": ["M3.9 bolt"],
    'it is .0625" tall': ['0.0625" tall'],
    "id should be half an inch, make it an inch wide": ["0.5 inch", "1 inch wide"],
    "a nine inch pipe on a four inch flange with twenty four bolts": ["9 inch pipe", "4 inch flange", "24 bolts"],
    "one leg measuring 1 inch and one measuring 2": ["one leg"],           # 'one' must NOT be converted here
    "12 3/4 pipe with a 20 1/2 flange": ["12.75 pipe", "20.5 flange"],
    "washer for an inch and a half screw": ["1.5 inch screw"],
    "an inch and a quarter wide": ["1.25 inch wide"],
    "gasket for 4 and a quarter flange": ["4.25 inch flange"],
    "it should have a dozen holes": ["12 holes"],
    "a 12 and a half inch wide gasket": ["12.5 inch wide"],
}
for src, must in _tests.items():
    out = normalize_text(src)
    assert all(m in out for m in must), f"normalizer test failed:\n  in : {src}\n  out: {out}\n  need: {must}"
assert tol_for(0.84) == NUM_TOL and tol_for(0.0118) < 0.001 and tol_for(0.0625) < NUM_TOL
print("normalizer self-tests passed")

## 3. Standard schema, catalog tables, train/val/test split
* One JSON shape for every part (same key names wherever a concept is shared, e.g. `thickness_in`).
* **Catalog tables** are built from every sheet row: known sizes (used to snap `0.84` → `0.8438`) and the washer size lookup (`M7` → its dimensions).
* With `GROUP_SPLIT = True`, a given pipe size / screw size never appears in both train and test, so test scores reflect genuinely unseen sizes.

In [ ]:
# ------------------------------------------------------------------ ONE STANDARD JSON SCHEMA FOR ALL PARTS
PART_TYPES = ["pipe_gasket", "washer", "bracket"]
SCHEMA = {                                             # key order here = key order in the JSON the model learns to write
    "pipe_gasket": ["od_in", "id_in", "thickness_in", "hole_count"],
    "washer":      ["screw_size", "od_in", "id_in", "thickness_in"],
    "bracket":     ["length_a_in", "length_b_in", "height_in", "thickness_in"],   # height_in = bracket width (your column name)
}
INT_FIELDS, STR_FIELDS = {"hole_count"}, {"screw_size"}
NUM_FIELDS = {pt: [f for f in fs if f not in STR_FIELDS] for pt, fs in SCHEMA.items()}

def canon_screw_size(s) -> str:
    """'No. 10'->'No. 10', 'M2.5'->'M2.5', '1 1/8"'->'1.125'  (same convention the normalized TEXT ends up with)"""
    s2 = normalize_text(str(s).replace('"', "").strip())
    m = re.fullmatch(r"(?i)m\s*([\d.]+)", s2)
    if m: return "M" + m.group(1)
    m = re.fullmatch(r"(?i)no\.?\s*(\d+)", s2)
    if m: return "No. " + m.group(1)
    return s2

def screw_in_text(sz, text_norm) -> bool:
    if sz.startswith("M"):  return re.search(rf"\b{re.escape(sz)}\b", text_norm, re.I) is not None
    if sz.startswith("No"): return re.search(rf"\bno\.? ?{sz[4:]}\b", text_norm, re.I) is not None
    return any(abs(float(sz) - n) <= 1e-3 for n in numbers_in(text_norm))

def coerce_value(field, v):
    """Turn whatever came back (str/float/None/junk) into a clean typed value, or None."""
    if v is None or (isinstance(v, float) and math.isnan(v)) or (isinstance(v, str) and not v.strip()):
        return None
    try:
        if field in STR_FIELDS: return canon_screw_size(v)
        x = float(normalize_text(v)) if isinstance(v, str) else float(v)
        if field in INT_FIELDS:
            return int(round(x)) if abs(x - round(x)) < 1e-9 else None
        return round(x, 4)
    except (ValueError, TypeError):
        return None

def row_to_params(r) -> dict:
    pt = r["part_type"]
    p = {"part_type": pt}
    for f in SCHEMA[pt]:
        p[f] = coerce_value(f, r[f])
    return p

def params_to_json(p: dict) -> str:
    """Canonical compact JSON (fixed key order) - this exact string is what the fine-tuned model learns to output."""
    ordered = {"part_type": p["part_type"], **{f: p.get(f) for f in SCHEMA[p["part_type"]]}}
    return json.dumps(ordered, separators=(",", ":"))

# ---- build the modelling table
data = df_all.copy()
data["text_norm"] = data["text"].map(normalize_text)
data["params"]    = data.apply(row_to_params, axis=1)
data["target"]    = data["params"].map(params_to_json)
assert data["params"].map(lambda p: all(v is not None for v in p.values())).all(), "some labelled rows have empty label cells"

# ---- catalog tables built from EVERY sheet row (including rows that have no text yet - geometry is still valid catalog data)
def _col(df, c): return pd.to_numeric(df[c], errors="coerce").dropna().round(4)

def _digits(v): return len(f"{v:.4f}".rstrip("0").split(".")[1]) if "." in f"{v:.4f}".rstrip("0") else 0

def canonical_values(values):
    """Your sheet stores some sizes at two precisions (4.62 AND 4.625, 0.84 AND 0.8438). Merge values that are within
    tol_for() of each other and keep the most precise one, so snapping has ONE catalog value per real size."""
    out, cluster = [], []
    for v in sorted(set(values)):
        if cluster and v - cluster[0] > tol_for(cluster[0]):
            out.append(max(cluster, key=lambda x: (_digits(x), x))); cluster = []
        cluster.append(v)
    if cluster: out.append(max(cluster, key=lambda x: (_digits(x), x)))
    return out

_gk, _wk, _bk = (raw_all[raw_all.part_type == pt] for pt in PART_TYPES)
STANDARDS = {                                           # known sizes per field, used to "snap" e.g. 0.84 -> 0.8438
    "pipe_gasket": {f: canonical_values(_col(_gk, f)) for f in ["od_in", "id_in", "thickness_in"]},
    "washer":      {f: canonical_values(_col(_wk, f)) for f in ["od_in", "id_in", "thickness_in"]},
    "bracket":     {f: canonical_values(_col(_bk, f)) for f in ["length_a_in", "length_b_in", "height_in", "thickness_in"]},
}
_dupe_sizes = sum(len(set(_col(d, f))) - len(canonical_values(_col(d, f))) for d, fs in [(_gk, ["od_in","id_in","thickness_in"]), (_wk, ["od_in","id_in","thickness_in"])] for f in fs)
print(f"label precision check: {_dupe_sizes} sizes appear at two different precisions in your sheet (e.g. 4.62 vs 4.625)")
WASHER_LOOKUP = {}                                      # 'M7' -> {od_in, id_in, thickness_in}: fills in a washer when only the size is given
for _, r in _wk.iterrows():
    WASHER_LOOKUP.setdefault(canon_screw_size(r["screw_size"]).lower(), {f: round(float(r[f]), 4) for f in ["od_in", "id_in", "thickness_in"]})
RANGES = {pt: {f: (float(min(v)), float(max(v))) for f, v in fs.items()} for pt, fs in STANDARDS.items()}
print("washer sizes in lookup table:", len(WASHER_LOOKUP), "| example M7 ->", WASHER_LOOKUP.get("m7"))

# ---- train / val / test split
def _group_key(r):
    p = r["params"]
    if r["part_type"] == "pipe_gasket": return ("g", round(p["od_in"], 1), round(p["id_in"], 1))
    if r["part_type"] == "washer":      return ("w", p["screw_size"])
    return ("b", p["length_a_in"], p["length_b_in"])

def make_split(df):
    rng = random.Random(SEED)
    df = df.copy(); df["group"] = df.apply(_group_key, axis=1) if GROUP_SPLIT else np.arange(len(df))
    df["split"] = "train"
    for pt, sub in df.groupby("part_type"):
        groups = sorted(set(sub["group"]), key=str); rng.shuffle(groups)
        n_test, n_val, got = TEST_FRAC * len(sub), VAL_FRAC * len(sub), {"test": 0, "val": 0}
        for g in groups:
            rows = sub.index[sub["group"] == g]
            if got["test"] < n_test:  s = "test"
            elif got["val"] < n_val:  s = "val"
            else:                     s = "train"
            if s != "train": got[s] += len(rows)
            df.loc[rows, "split"] = s
    return df

flagged_ids = set()    # filled in by the audit cell below
data = make_split(data)
print(data.groupby(["part_type", "split"]).size().unstack(fill_value=0))
assert set(data.loc[data.split == "train", "part_type"]) == set(PART_TYPES), "train split is missing a part type"

## 4. Audit — do the labels agree with their own text?
A label whose numbers don't appear in its text teaches the model to invent values. The list below gives the sheet name and **Excel row** of each suspect row so you can fix it in the spreadsheet (or set `EXCLUDE_AUDIT_FLAGGED = True`).

In [ ]:
# ------------------------------------------------------------------ AUDIT: do the label numbers actually appear in the text?
def audit_row(r):
    nums, bad = numbers_in(r["text_norm"]), []
    for f in NUM_FIELDS[r["part_type"]]:
        v = r["params"][f]
        ok = (int(v) in [int(n) for n in nums if n == int(n)]) if f in INT_FIELDS else any(abs(v - n) <= tol_for(v) for n in nums)
        if not ok: bad.append((f, v))
    if r["part_type"] == "washer":
        sz = r["params"]["screw_size"]
        if not screw_in_text(sz, r["text_norm"]): bad.append(("screw_size", sz))
    return bad

data["audit_bad"] = data.apply(audit_row, axis=1)
audit_report = data[data.audit_bad.map(len) > 0][["sheet", "sheet_row", "text", "audit_bad"]].copy()
audit_report["numbers_in_text"] = data.loc[audit_report.index, "text_norm"].map(numbers_in)
pd.set_option("display.max_colwidth", 120, "display.width", 220)
print(f"{len(audit_report)} of {len(data)} rows have a label value that is NOT in their own text.")
print("Fix these in the spreadsheet (sheet + Excel row shown), or set EXCLUDE_AUDIT_FLAGGED = True.\n")
show_df(audit_report)

dups = data[data.duplicated("text_norm", keep=False)]
print(f"\nduplicate texts after normalization: {len(dups)}")

if EXCLUDE_AUDIT_FLAGGED and len(audit_report):
    data = data.drop(index=audit_report.index).reset_index(drop=True)
    print(f"dropped {len(audit_report)} flagged rows; {len(data)} remain")
    data = make_split(data.drop(columns=["split", "group"]))
train_df, val_df, test_df = (data[data.split == s].reset_index(drop=True) for s in ["train", "val", "test"])
print(f"\ntrain {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

## 5. Validation layer + CAD generator (deterministic — no ML)
**Validation** turns the model's JSON into the table the user sees. Each value gets a status:

| status | meaning |
|---|---|
| ✅ extracted | the model found it and the number really is in your text |
| ⚠️ not found in your text | the model produced it but it isn't in your description (possible hallucination) |
| ❌ missing | not extracted |
| 🔷 snapped / standard size | matched to a catalog size, or filled from the washer size table (`M7` alone is enough) — values you state explicitly are never overwritten |
| 🟨 assumed default | needed for CAD but not in your data (gasket bolt-circle diameter and bolt-hole size) — editable |
| ✏️ edited | changed by the user |

Hard rules (positive values, OD > ID, bolt holes inside the ring, …) block CAD generation; range warnings don't.

In [ ]:
# ------------------------------------------------------------------ STAGE 3: VALIDATION (rules first, optional off-the-shelf LLM second)
# Pipeline box: "Validation checks for negative values, missing data, out of range, etc."
STATUS_LABEL = {
    "extracted":   "✅ extracted",
    "missing":     "❌ missing",
    "not_in_text": "⚠️ not found in your text",
    "snapped":     "🔷 snapped to catalog size",
    "lookup":      "🔷 from standard-size table",
    "default":     "🟨 assumed default",
    "edited":      "✏️ edited by you",
    "llm_flag":    "⚠️ verifier disagrees",
}
DERIVED_GASKET = ["bolt_circle_in", "bolt_hole_in"]      # not in the dataset -> derived defaults the user can override

def fmt_value(v):
    if v is None: return ""
    if isinstance(v, int) and not isinstance(v, bool): return str(v)
    if isinstance(v, float): return dec(v)
    return str(v)

def snap_value(part_type, field, v):
    if not SNAP_TO_STANDARD or field not in STANDARDS[part_type]: return v
    best = min(STANDARDS[part_type][field], key=lambda s: abs(s - v))
    return best if abs(best - v) <= tol_for(v) else v

def default_bolt_geometry(od, idd):
    if od <= idd: return None, None                               # invalid OD/ID: the rules will report it
    w = (od - idd) / 2                                            # radial width of the gasket ring
    return round((od + idd) / 2, 4), round(min(max(0.4 * w, 0.19), 1.25, 0.6 * w), 4)

def make_table(part_type, text_norm, raw_params):
    """model output (dict or None) -> list of row dicts {field, value, status, note}"""
    nums, raw = numbers_in(text_norm), (raw_params or {})
    rows = [{"field": f, "value": coerce_value(f, raw.get(f)), "status": "extracted", "note": ""} for f in SCHEMA[part_type]]
    R = {r["field"]: r for r in rows}
    for r in rows:
        if r["value"] is None: r.update(status="missing", note="the model did not find this in your text")

    std = WASHER_LOOKUP.get((R["screw_size"]["value"] or "").lower()) if part_type == "washer" else None
    if std:                                                        # fill only what's missing (partial overrides survive)
        for f, sv in std.items():
            if R[f]["value"] is None:
                R[f].update(value=sv, status="lookup", note=f"standard size for {R['screw_size']['value']}")

    for f in SCHEMA[part_type]:                                    # grounding check + snapping for model-extracted values
        r, v = R[f], R[f]["value"]
        if r["status"] != "extracted": continue
        if f in STR_FIELDS: grounded = screw_in_text(v, text_norm)
        elif f in INT_FIELDS: grounded = v in [int(n) for n in nums if n == int(n)]
        else: grounded = any(abs(v - n) <= tol_for(v) for n in nums)
        if not grounded:
            r.update(status="not_in_text", note="this value is not in your description - please check it"); continue
        if f not in STR_FIELDS and f not in INT_FIELDS:
            sv = snap_value(part_type, f, v)
            if sv != v: r.update(value=sv, status="snapped", note=f"you wrote ~{dec(v)}; catalog size is {dec(sv)}")
        if std and f in std and abs(R[f]["value"] - std[f]) > tol_for(std[f]):
            r["note"] = (r["note"] + " | " if r["note"] else "") + f"differs from standard {R['screw_size']['value']} ({dec(std[f])})"

    if part_type == "pipe_gasket" and R["od_in"]["value"] and R["id_in"]["value"]:
        bc, hd = default_bolt_geometry(R["od_in"]["value"], R["id_in"]["value"])
        rows += [{"field": "bolt_circle_in", "value": bc, "status": "default", "note": "assumed: halfway between ID and OD (not in your data)"},
                 {"field": "bolt_hole_in",   "value": hd, "status": "default", "note": "assumed hole size (not in your data)"}]
    return rows

def rows_to_params(part_type, rows):
    p = {"part_type": part_type}
    for r in rows: p[r["field"]] = coerce_value(r["field"], r["value"])
    return p

def apply_edits(part_type, old_rows, ui_values):
    """ui_values: {field: text typed in the table}. Only changed cells get 'edited'; everything else keeps its status."""
    rows, edited = copy.deepcopy(old_rows), set()
    for r in rows:
        if r["field"] not in ui_values: continue
        nv = coerce_value(r["field"], ui_values[r["field"]])
        if nv != r["value"]:
            r.update(value=nv, status="edited" if nv is not None else "missing", note="changed by you" if nv is not None else "cleared")
            edited.add(r["field"])
    R = {r["field"]: r for r in rows}
    if part_type == "washer" and R["screw_size"]["value"]:         # user typed a size -> fill blanks from the standard table
        std = WASHER_LOOKUP.get(R["screw_size"]["value"].lower())
        for f, sv in (std or {}).items():
            if R[f]["value"] is None: R[f].update(value=sv, status="lookup", note=f"standard size for {R['screw_size']['value']}")
    if part_type == "pipe_gasket" and R["od_in"]["value"] and R["id_in"]["value"]:
        bc, hd = default_bolt_geometry(R["od_in"]["value"], R["id_in"]["value"])
        for f, dv in zip(DERIVED_GASKET, (bc, hd)):                # follow OD/ID edits unless the user set them
            if f not in R: rows.append({"field": f, "value": dv, "status": "default", "note": "assumed (not in your data)"}); R[f] = rows[-1]
            elif f not in edited and R[f]["status"] == "default": R[f]["value"] = dv
    return rows, edited

def check_rules(part_type, p):
    """hard errors block CAD generation; warnings are just shown"""
    issues = []
    err  = lambda f, m: issues.append({"level": "error", "field": f, "msg": m})
    warn = lambda f, m: issues.append({"level": "warning", "field": f, "msg": m})
    def check_present(fields):
        for f in fields:
            v = p.get(f)
            if v is None: err(f, "missing - fill it in the table")
            elif f not in STR_FIELDS and v <= 0: err(f, "must be greater than zero")
    check_present(SCHEMA[part_type])
    if issues: return issues
    if part_type in ("pipe_gasket", "washer") and p["od_in"] <= p["id_in"]:
        err("od_in", "outer diameter must be larger than inner diameter"); return issues
    if part_type == "bracket" and p["thickness_in"] >= min(p["length_a_in"], p["length_b_in"]):
        err("thickness_in", "thickness must be smaller than both leg lengths"); return issues
    if part_type == "pipe_gasket":
        check_present(DERIVED_GASKET)
        if issues: return issues
        n, bc, hd = p["hole_count"], p["bolt_circle_in"], p["bolt_hole_in"]
        if not 1 <= n <= 64: err("hole_count", "expected between 1 and 64 bolt holes"); return issues
        if bc / 2 - hd / 2 <= p["id_in"] / 2 or bc / 2 + hd / 2 >= p["od_in"] / 2:
            err("bolt_circle_in", "bolt holes would fall outside the gasket ring - adjust bolt circle / hole size"); return issues
        if n >= 2 and bc * math.sin(math.pi / n) < 1.1 * hd:
            warn("hole_count", "bolt holes nearly overlap - adjust bolt circle / hole size")
    for f, (lo, hi) in RANGES[part_type].items():
        if p[f] < 0.5 * lo or p[f] > 2 * hi: warn(f, f"unusual size (catalog range is {dec(lo)} to {dec(hi)} in)")
    return issues

def llm_verify(part_type, text_norm, params):
    """OPTIONAL off-the-shelf check: ask the UNTUNED base model which values look wrong. Small models are weak verifiers -
    that's why the deterministic rules above do the heavy lifting and this is off by default."""
    shown = {k: params[k] for k in SCHEMA[part_type] if params.get(k) is not None}
    prompt = (f"Description: {text_norm}\nExtracted values: {json.dumps(shown)}\n"
              "List the keys whose value does NOT match the description, as a JSON list ([] if all match).\nAnswer: ")
    out = generate([prompt], use_adapter=False, max_new_tokens=40)[0]
    m = re.search(r"\[[^\]]*\]", out)
    try:    return [k for k in json.loads(m.group(0)) if k in SCHEMA[part_type]] if m else []
    except Exception: return []

def rows_to_df(rows):
    return pd.DataFrame([[r["field"], fmt_value(r["value"]), STATUS_LABEL[r["status"]], r["note"]] for r in rows],
                        columns=["field", "value", "status", "note"])

**CAD:** parameters (inches) → CadQuery solid → scaled ×25.4 → STL in mm. The self-test checks that volumes match the maths, which proves the bolt holes are really cut.

In [ ]:
# ------------------------------------------------------------------ STAGE 4: JSON -> CAD -> STL  (deterministic, no ML)
import cadquery as cq, trimesh
IN_TO_MM = 25.4        # STL has no units and slicers assume mm, so geometry built in inches is scaled once, at export

def build_gasket(p):
    od, idd, t, n = p["od_in"], p["id_in"], p["thickness_in"], int(p["hole_count"])
    bc, hd = p["bolt_circle_in"], p["bolt_hole_in"]
    ring = cq.Workplane("XY").circle(od / 2).circle(idd / 2).extrude(t)
    pts = [((bc / 2) * math.cos(2 * math.pi * i / n), (bc / 2) * math.sin(2 * math.pi * i / n)) for i in range(n)]
    holes = cq.Workplane("XY").workplane(offset=-t).pushPoints(pts).circle(hd / 2).extrude(3 * t)   # oversized so it cuts clean through
    return ring.cut(holes)

def build_washer(p):
    return cq.Workplane("XY").circle(p["od_in"] / 2).circle(p["id_in"] / 2).extrude(p["thickness_in"])

def build_bracket(p):
    a, b, w, t = p["length_a_in"], p["length_b_in"], p["height_in"], p["thickness_in"]
    return cq.Workplane("XY").polyline([(0, 0), (a, 0), (a, t), (t, t), (t, b), (0, b)]).close().extrude(w)   # L profile, extruded by the width

BUILDERS = {"pipe_gasket": build_gasket, "washer": build_washer, "bracket": build_bracket}

def build_stl(params: dict, out_path: str) -> dict:
    solid = BUILDERS[params["part_type"]](params).val().scale(IN_TO_MM)
    cq.exporters.export(solid, str(out_path), tolerance=0.01, angularTolerance=0.05)
    m = trimesh.load(str(out_path))
    return {"watertight": bool(m.is_watertight), "volume_mm3": round(float(m.volume), 2), "size_mm": [round(float(x), 2) for x in m.extents]}

# ---- tests: geometry must match the maths (this is how we know the holes were REALLY cut)
def _cuin(x): return x * IN_TO_MM ** 3
_g = {"part_type": "pipe_gasket", "od_in": 3.5, "id_in": 0.8438, "thickness_in": 0.0625, "hole_count": 4, "bolt_circle_in": 2.1719, "bolt_hole_in": 0.5325}
_i = build_stl(_g, WORKDIR / "test_gasket.stl")
_expect = _cuin(math.pi / 4 * (3.5**2 - 0.8438**2) * 0.0625 - 4 * math.pi / 4 * 0.5325**2 * 0.0625)
assert _i["watertight"] and abs(_i["volume_mm3"] - _expect) / _expect < 0.01, (_i, _expect)
_w = {"part_type": "washer", "od_in": 0.5512, "id_in": 0.2913, "thickness_in": 0.063}
_i = build_stl(_w, WORKDIR / "test_washer.stl")
assert _i["watertight"] and abs(_i["size_mm"][0] - 0.5512 * 25.4) < 0.05, _i
_b = {"part_type": "bracket", "length_a_in": 2.0, "length_b_in": 1.5, "height_in": 0.5, "thickness_in": 0.125}
_i = build_stl(_b, WORKDIR / "test_bracket.stl")
_expect = _cuin((2.0 * 0.125 + (1.5 - 0.125) * 0.125) * 0.5)
assert _i["watertight"] and abs(_i["volume_mm3"] - _expect) / _expect < 0.01 and sorted(_i["size_mm"], reverse=True)[0] == round(2.0 * 25.4, 2), (_i, _expect)
print("CAD self-tests passed (watertight, volumes match the maths, holes really cut, mm scaling correct)")

## 6. Stage 1 — part-type classifier, trained from scratch
A bag-of-words neural net (learned embeddings → small MLP) with **randomly initialised weights**: no pretrained model is involved.

In [ ]:
# ------------------------------------------------------------------ STAGE 1: part-type classifier, TRAINED FROM SCRATCH
# Pipeline box: "Trained from scratch model extracts part type".
# A small neural net with randomly-initialised weights - no pretrained model is used anywhere in this stage.
from collections import Counter

def clf_tokens(text_norm: str):
    t = re.sub(r"(?<![\w.])\d+(?:\.\d+)?", " <num> ", text_norm.lower())      # numbers don't help decide the part type
    words = re.findall(r"<num>|[a-z]+", t)
    return words + [a + "_" + b for a, b in zip(words, words[1:])]              # unigrams + bigrams

PT2ID = {pt: i for i, pt in enumerate(PART_TYPES)}
_doc_freq = Counter(tok for t in train_df.text_norm for tok in set(clf_tokens(t)))
VOCAB = {"<unk>": 0}
for _w, _c in sorted(_doc_freq.items()):
    if _c >= 2: VOCAB[_w] = len(VOCAB)                                          # only tokens seen in >=2 training texts

def clf_encode(text_norm: str):
    return [VOCAB.get(t, 0) for t in clf_tokens(text_norm)] or [0]

print(f"classifier vocabulary: {len(VOCAB)} tokens")

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

class BowClassifier(nn.Module):
    """token ids -> mean of learned embeddings -> small MLP -> part type"""
    def __init__(self, vocab_size, n_classes, dim=48, hidden=64, p=0.3):
        super().__init__()
        self.emb = nn.EmbeddingBag(vocab_size, dim, mode="mean")
        self.net = nn.Sequential(nn.Dropout(p), nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(p), nn.Linear(hidden, n_classes))
    def forward(self, ids, offsets):
        return self.net(self.emb(ids, offsets))

def make_batch(texts):
    seqs = [clf_encode(t) for t in texts]
    offsets = torch.tensor([0] + list(np.cumsum([len(s) for s in seqs])[:-1]), dtype=torch.long)
    flat = torch.tensor([i for s in seqs for i in s], dtype=torch.long)
    return flat.to(DEVICE), offsets.to(DEVICE)

@torch.no_grad()
def clf_predict_ids(texts):
    clf.eval(); flat, off = make_batch(texts)
    return F.softmax(clf(flat, off), dim=-1).cpu().numpy()

def clf_accuracy(df):
    pred = clf_predict_ids(df.text_norm.tolist()).argmax(1)
    return float((pred == df.part_type.map(PT2ID).values).mean())

clf = BowClassifier(len(VOCAB), len(PART_TYPES)).to(DEVICE)
opt = torch.optim.AdamW(clf.parameters(), lr=5e-3, weight_decay=1e-3)
y_train = torch.tensor(train_df.part_type.map(PT2ID).values, dtype=torch.long, device=DEVICE)
best_acc, best_state = -1.0, None

for epoch in range(1, 81):
    clf.train()
    perm = np.random.permutation(len(train_df))
    for i in range(0, len(perm), 32):
        idx = perm[i:i + 32]
        flat, off = make_batch(train_df.text_norm.iloc[idx].tolist())
        loss = F.cross_entropy(clf(flat, off), y_train[torch.as_tensor(idx, device=DEVICE)])
        opt.zero_grad(); loss.backward(); opt.step()
    va = clf_accuracy(val_df)
    if va >= best_acc: best_acc, best_state = va, copy.deepcopy(clf.state_dict())
    if epoch % 20 == 0: print(f"epoch {epoch:3d}  last-batch loss {loss.item():.3f}  val acc {va:.3f}")
clf.load_state_dict(best_state)

print(f"\nbest val accuracy {best_acc:.3f}  |  TEST accuracy {clf_accuracy(test_df):.3f}")
_pred = clf_predict_ids(test_df.text_norm.tolist()).argmax(1)
print(classification_report(test_df.part_type.map(PT2ID), _pred, target_names=PART_TYPES, zero_division=0))
print(pd.DataFrame(confusion_matrix(test_df.part_type.map(PT2ID), _pred, labels=range(len(PART_TYPES))),
                   index=[f"true {p}" for p in PART_TYPES], columns=[f"pred {p}" for p in PART_TYPES]))

def predict_part_type(text_norm: str):
    probs = clf_predict_ids([text_norm])[0]
    return PART_TYPES[int(probs.argmax())], {pt: float(p) for pt, p in zip(PART_TYPES, probs)}

## 7. Stage 2 — parameter extractor, fine-tuned with LoRA
Fine-tunes a small open LLM to write the standard-schema JSON. Loss is computed only on the JSON answer. The un-tuned base model doubles as the **off-the-shelf baseline** (adapter switched off), so the report can show what fine-tuning bought you.  
`MODEL_NAME` defaults to Qwen2.5-1.5B-Instruct; use the 0.5B version for a faster run.

In [ ]:
# ------------------------------------------------------------------ STAGE 2 (prompts + parsing): fine-tuned parameter extractor
# Pipeline boxes: "Fine tuned model extracts parameters" -> "Outputs JSON with standard schema for all parts"
SCHEMA_DOC = "\n".join(f"{pt}: " + ", ".join(["part_type"] + fs) for pt, fs in SCHEMA.items())

def ft_prompt(part_type, text_norm):
    """short prompt for the FINE-TUNED model (it has learned the schema, so it doesn't need to be told)"""
    return f"Part type: {part_type}\nDescription: {text_norm}\nJSON:\n"

_clean_train = train_df[train_df.audit_bad.map(len) == 0]                     # few-shot examples must not be mislabelled rows
_FEWSHOT = {pt: _clean_train[_clean_train.part_type == pt].iloc[0] for pt in PART_TYPES}
def baseline_prompt(part_type, text_norm):
    """long prompt for the OFF-THE-SHELF baseline (schema + one worked example per part type, no fine-tuning)"""
    shots = "".join(f"Part type: {pt}\nDescription: {r.text_norm}\nJSON:\n{r.target}\n\n" for pt, r in _FEWSHOT.items())
    return ("Convert a description of a mechanical part into compact JSON. Use exactly these keys, in this order:\n" + SCHEMA_DOC +
            "\nAll dimensions are decimal inches. hole_count is an integer. Output only the JSON.\n\n" + shots +
            f"Part type: {part_type}\nDescription: {text_norm}\nJSON:\n")

def parse_json_obj(s):
    """first {...} in the model's text -> dict, else None (flat JSON, so first closing brace ends it)"""
    m = re.search(r"\{.*?\}", s or "", re.S)
    if not m: return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

# ---- scoring helpers (used by the evaluation cell)
def field_ok(f, pred, true):
    p = coerce_value(f, pred)
    if p is None: return False
    if f in STR_FIELDS: return p.lower() == str(true).lower()
    if f in INT_FIELDS: return p == true
    return abs(p - true) <= tol_for(true)

def score_outputs(df, outs):
    recs = []
    for r, o in zip(df.itertuples(), outs):
        obj = parse_json_obj(o)
        oks = {f: bool(obj) and field_ok(f, obj.get(f), r.params[f]) for f in SCHEMA[r.part_type]}
        recs.append({"part_type": r.part_type, "json_valid": obj is not None, **oks, "all_correct": all(oks.values())})
    return pd.DataFrame(recs)

def summarize(scores, name):
    fields = [c for c in scores.columns if c not in ("part_type", "json_valid", "all_correct")]
    row = {"model": name, "JSON valid": scores.json_valid.mean(), **{f: scores[f].astype(float).mean() for f in fields}, "ALL fields correct": scores.all_correct.mean()}
    return pd.Series(row)

_t = train_df.iloc[:3]
_ok = score_outputs(_t, [r.target + " trailing text" for r in _t.itertuples()])
assert _ok.all_correct.all() and _ok.json_valid.all(), "scoring helpers broken"
print("prompt/parse/score helpers ready | example fine-tune prompt:\n" + ft_prompt("washer", "washer for an M7 screw") + "{...}")

In [ ]:
# Fine-tunes a small open LLM with LoRA so that:  "Part type + normalized description"  ->  standard-schema JSON.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_CUDA = DEVICE == "cuda"
USE_BF16 = USE_CUDA and torch.cuda.is_bf16_supported()
LOAD_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32     # T4 has no bf16: use plain fp32 (a bit slower, but no fp16 NaN/overflow risk)
AMP_DTYPE, AMP_ON = torch.bfloat16, USE_BF16                     # mixed precision only on GPUs that support bf16 (L4/A100/H100)
if not USE_CUDA: print("WARNING: no GPU found. Runtime > Change runtime type > T4 GPU (CPU training will be very slow).")
torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token
try:    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=LOAD_DTYPE)
except TypeError: base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=LOAD_DTYPE)   # older transformers
base.to(DEVICE)
model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

# ---- tokenize: loss is computed ONLY on the JSON answer, not on the prompt
MAX_LEN = 256
def encode_example(part_type, text_norm, target):
    p = tok(ft_prompt(part_type, text_norm), add_special_tokens=False)["input_ids"]
    c = tok(target + tok.eos_token, add_special_tokens=False)["input_ids"]
    return (p + c)[:MAX_LEN], ([-100] * len(p) + c)[:MAX_LEN]
def encode_df(df): return [encode_example(r.part_type, r.text_norm, r.target) for r in df.itertuples()]
train_enc, val_enc = encode_df(train_df), encode_df(val_df)
print("longest training example:", max(len(i) for i, _ in train_enc), "tokens (MAX_LEN =", MAX_LEN, ")")

def collate(batch):
    L = max(len(i) for i, _ in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for k, (i, l) in enumerate(batch):
        ids[k, :len(i)] = torch.tensor(i); lab[k, :len(l)] = torch.tensor(l); att[k, :len(i)] = 1
    return ids.to(DEVICE), att.to(DEVICE), lab.to(DEVICE)

def eval_loss(enc):
    model.eval(); total, n = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(enc), BATCH_SIZE):
            ids, att, lab = collate(enc[i:i + BATCH_SIZE])
            with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=AMP_ON):
                out = model(input_ids=ids, attention_mask=att, labels=lab)
            k = int((lab[:, 1:] != -100).sum()); total += out.loss.item() * k; n += k
    return total / n

# ---- training loop (plain PyTorch, so it doesn't depend on Trainer/TRL API changes)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LORA_LR, weight_decay=0.0)
steps_per_epoch = math.ceil(len(train_enc) / BATCH_SIZE)
total_steps = LORA_EPOCHS * steps_per_epoch; warmup = max(1, int(0.05 * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup) * max(0.0, 1 - s / total_steps))

best_val, best_state, history = float("inf"), None, []
for epoch in range(1, LORA_EPOCHS + 1):
    model.train(); order = np.random.permutation(len(train_enc)); running = 0.0
    for i in range(0, len(order), BATCH_SIZE):
        ids, att, lab = collate([train_enc[j] for j in order[i:i + BATCH_SIZE]])
        with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=AMP_ON):
            loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step(); sched.step(); running += loss.item()
    val_loss = eval_loss(val_enc); history.append((epoch, running / steps_per_epoch, val_loss))
    print(f"epoch {epoch}/{LORA_EPOCHS}  train loss {running / steps_per_epoch:.4f}  val loss {val_loss:.4f}")
    if val_loss < best_val:                                   # keep the best epoch (tiny datasets overfit quickly)
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
model.load_state_dict(best_state, strict=False)
model.eval()
print(f"restored best epoch (val loss {best_val:.4f})")

@torch.no_grad()
def generate(prompts, use_adapter=True, max_new_tokens=96, batch=8):
    """use_adapter=False runs the ORIGINAL, un-fine-tuned model (that's our off-the-shelf baseline / verifier)"""
    model.eval(); outs = []; tok.padding_side = "left"
    for i in range(0, len(prompts), batch):
        enc = tok(prompts[i:i + batch], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        ctx = nullcontext() if use_adapter else model.disable_adapter()
        with ctx, torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=AMP_ON):
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tok.padding_side = "right"
    return outs

def extract_params(part_type, text_norm):
    """Pipeline box 'Fine tuned model extracts parameters': returns a dict, or None if the output wasn't valid JSON."""
    return parse_json_obj(generate([ft_prompt(part_type, text_norm)])[0])

_demo = test_df.iloc[0]
print("\nexample ->", _demo.text_norm, "\n  model :", extract_params(_demo.part_type, _demo.text_norm), "\n  label :", _demo.target)

## 8. Evaluation on the held-out test set
Fine-tuned vs off-the-shelf few-shot, per-field accuracy, and the full classifier → extractor chain. Mistakes on rows the audit flagged are probably label errors, not model errors.

In [ ]:
# ------------------------------------------------------------------ EVALUATION on the held-out TEST split
# "Correct" = within tol_for(value) of the label (labels are catalog-snapped, texts are often rounded), hole_count/screw_size exact.
def run_eval(df, prompt_fn, use_adapter):
    outs = generate([prompt_fn(r.part_type, r.text_norm) for r in df.itertuples()], use_adapter=use_adapter)
    return score_outputs(df, outs), outs

sc_ft,   outs_ft   = run_eval(test_df, ft_prompt, True)
sc_base, outs_base = run_eval(test_df, baseline_prompt, False)
comparison = pd.DataFrame([summarize(sc_ft, "fine-tuned (LoRA)"), summarize(sc_base, "off-the-shelf, few-shot")]).set_index("model")
print("Extractor accuracy on the test set (true part type given, so this isolates the extractor):")
show_df(comparison.round(3))
print("\nfine-tuned, ALL fields correct by part type:")
show_df(sc_ft.groupby("part_type").all_correct.agg(["mean", "size"]).round(3))

# ---- end-to-end: classifier -> extractor (what a real user experiences)
pred_types = [predict_part_type(t)[0] for t in test_df.text_norm]
type_ok = np.array(pred_types) == test_df.part_type.values
e2e = score_outputs(test_df, generate([ft_prompt(pt, t) for pt, t in zip(pred_types, test_df.text_norm)]))
print(f"\nend-to-end on test: part type right {type_ok.mean():.1%} | every field right {(e2e.all_correct.values & type_ok).mean():.1%}")

# ---- look at the mistakes (rows the audit flagged probably have wrong LABELS, not wrong predictions)
bad = np.where(~sc_ft.all_correct.values)[0]
for i in bad[:8]:
    r = test_df.iloc[i]
    print(f"\n[{'label likely wrong' if r.audit_bad else 'model error'}] {r.text}\n   expected: {r.target}\n   got     : {parse_json_obj(outs_ft[i])}")

## 9. Pipeline glue
Text → part type → parameters → table, and the confirm step (apply edits → re-validate → CAD → log feedback).

In [ ]:
# ------------------------------------------------------------------ PIPELINE GLUE: text -> type -> parameters -> validated table
FEEDBACK_LOG = WORKDIR / "feedback.jsonl"

def analyze(text: str, forced_type: str = None) -> dict:
    """Boxes: preprocess -> classifier (from scratch) -> fine-tuned extractor -> JSON -> validation -> table."""
    text_norm = normalize_text(text)
    pt, probs = (forced_type, None) if forced_type else predict_part_type(text_norm)
    raw = extract_params(pt, text_norm)                       # dict, or None if the model's output wasn't valid JSON
    rows = make_table(pt, text_norm, raw)
    if USE_LLM_VERIFIER:
        suspects = llm_verify(pt, text_norm, rows_to_params(pt, rows))
        for r in rows:
            if r["field"] in suspects and r["status"] in ("extracted", "snapped"):
                r.update(status="llm_flag", note="the off-the-shelf verifier thinks this may be wrong")
    return {"text": text, "text_norm": text_norm, "part_type": pt, "probs": probs, "raw": raw, "rows": rows}

def confirm_and_build(state: dict, ui_values: dict):
    """User pressed Confirm: apply their edits -> re-validate -> (if OK) JSON goes to the CAD script -> STL."""
    pt = state["part_type"]
    rows, edited = apply_edits(pt, state["rows"], ui_values)
    params = rows_to_params(pt, rows)
    issues = check_rules(pt, params)
    result = {"rows": rows, "params": params, "issues": issues, "stl_path": None, "info": None}
    if any(i["level"] == "error" for i in issues):
        return result
    path = WORKDIR / f"{pt}_{int(time.time())}.stl"
    result["info"] = build_stl(params, path)
    result["stl_path"] = str(path)
    with open(FEEDBACK_LOG, "a") as fh:                       # feedback loop: (text, model output, what the user corrected)
        fh.write(json.dumps({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "text": state["text"], "text_norm": state["text_norm"],
                             "predicted_type": pt, "model_output": state["raw"], "final_params": params,
                             "user_edited_fields": sorted(edited)}) + "\n")
    return result

## 10. The app (plain HTML)
A single HTML/CSS/JS page (`INDEX_HTML`) served by a small Flask backend. Extract → review the color-coded table → edit any cell in the **value** column → Confirm → 3D preview (three.js), download, or start over.  
Changing the *part type* dropdown re-runs the fine-tuned extractor for that type. The page and the four `/api/...` endpoints are ordinary code you can move to any server, including a Hugging Face Docker Space.

In [ ]:
# ------------------------------------------------------------------ GUI: one plain HTML page + a tiny Flask backend (no Gradio)
# The page (INDEX_HTML) is ordinary HTML/CSS/JS. It talks to four JSON endpoints that call the pipeline functions above.
# ---- the app needs the two trained models from sections 6 and 7; fail early with a clear message if they aren't ready
_not_ready = [n for n in ("predict_part_type", "extract_params") if n not in globals()]
if _not_ready:
    raise RuntimeError(f"Not ready: {_not_ready} not defined. Run the classifier cell (section 6) and the LoRA fine-tuning cell "
                       "(section 7) successfully first - the app uses the models they create.")

import uuid, threading
from flask import Flask, jsonify, request, send_file, abort
from werkzeug.serving import make_server

INDEX_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Text to STL</title>
<style>
  body{font-family:system-ui,sans-serif;margin:0;background:#f4f5f7;color:#1c1e21}
  main{max-width:960px;margin:0 auto;padding:20px}
  h1{margin:0 0 4px} .sub{color:#555;margin:0 0 16px}
  .card{background:#fff;border-radius:10px;padding:16px;margin-bottom:14px;box-shadow:0 1px 3px rgba(0,0,0,.12)}
  textarea{width:100%;box-sizing:border-box;font:inherit;padding:8px;border:1px solid #bbb;border-radius:6px}
  button,select{font:inherit;padding:8px 14px;border-radius:6px;border:1px solid #888;background:#fff;cursor:pointer}
  button.primary{background:#1a5fd0;color:#fff;border-color:#1a5fd0} button:disabled{opacity:.45;cursor:default}
  .row{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin-top:10px}
  table{border-collapse:collapse;width:100%} th,td{text-align:left;padding:6px 8px;border-bottom:1px solid #e3e3e3}
  td input{width:130px;font:inherit;padding:4px 6px;border:1px solid #aaa;border-radius:4px}
  tr.extracted td.st{background:#dff3e0} tr.missing td.st{background:#f9d6d5} tr.not_in_text td.st,tr.llm_flag td.st{background:#fdebc4}
  tr.snapped td.st,tr.lookup td.st{background:#d8e7fb} tr.default td.st{background:#fff6b8} tr.edited td.st{background:#e7dcf7}
  .legend span{display:inline-block;padding:2px 8px;margin:2px 4px 2px 0;border-radius:4px;font-size:13px}
  #msg{white-space:pre-wrap} .err{color:#b3261e} .ok{color:#1b6e2d}
  #view{width:100%;height:340px;background:#eceff3;border-radius:8px;display:flex;align-items:center;justify-content:center;color:#666}
  pre{background:#f0f1f3;padding:8px;border-radius:6px;overflow:auto;font-size:13px;margin:8px 0 0}
</style></head><body><main>
<h1>Text &rarr; STL</h1>
<p class="sub">Describe a <b>pipe gasket, washer/spacer, or L-bracket</b>. Fractions and decimals both work.</p>

<div class="card">
  <textarea id="text" rows="3" placeholder="e.g. gasket for a 6 inch flange on a 2 3/8 pipe, 1/16 thick with four bolt holes"></textarea>
  <div class="row"><button class="primary" id="bExtract">Extract parameters</button><button id="bReset">Start over</button></div>
</div>

<div class="card" id="tableCard" hidden>
  <div class="row" style="margin-top:0"><label>Part type <select id="ptype"></select></label>
    <span style="color:#666;font-size:13px">changing it re-runs the extractor for that type</span></div>
  <p id="info"></p>
  <table><thead><tr><th>field</th><th>value (editable)</th><th>status</th><th>note</th></tr></thead><tbody id="rows"></tbody></table>
  <div class="legend" style="margin-top:10px">
    <span style="background:#dff3e0">found in your text</span><span style="background:#fdebc4">not in your text</span>
    <span style="background:#f9d6d5">missing</span><span style="background:#d8e7fb">catalog size</span>
    <span style="background:#fff6b8">assumed default</span><span style="background:#e7dcf7">edited by you</span></div>
  <div class="row"><button class="primary" id="bConfirm">Confirm &amp; generate STL</button></div>
  <div id="msg"></div>
</div>

<div class="card" id="outCard" hidden>
  <div id="view">3D preview</div>
  <div class="row"><a id="dl" href="#"><button>Download STL</button></a></div>
  <details><summary>JSON sent to the CAD script</summary><pre id="json"></pre></details>
</div>
</main>

<script type="importmap">{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/"}}</script>
<script type="module">
import * as THREE from 'three';
import {STLLoader} from 'three/addons/loaders/STLLoader.js';
import {OrbitControls} from 'three/addons/controls/OrbitControls.js';

const $ = id => document.getElementById(id);
let sid = null, renderer = null, raf = 0;
const TYPES = ['pipe_gasket','washer','bracket'];
TYPES.forEach(t => { const o = document.createElement('option'); o.value = o.textContent = t; $('ptype').appendChild(o); });

async function api(path, body){
  const r = await fetch(path, {method:'POST', headers:{'Content-Type':'application/json'}, body:JSON.stringify(body)});
  const j = await r.json(); if(!r.ok) throw new Error(j.error || r.statusText); return j;
}
function say(text, cls){ const m = $('msg'); m.textContent = text; m.className = cls || ''; }
function showRows(rows){
  const tb = $('rows'); tb.textContent = '';
  rows.forEach(r => {
    const tr = document.createElement('tr'); tr.className = r.status;
    const f = document.createElement('td'); f.textContent = r.field;
    const v = document.createElement('td'); const i = document.createElement('input'); i.value = r.value; i.dataset.field = r.field; v.appendChild(i);
    const s = document.createElement('td'); s.className = 'st'; s.textContent = r.label;
    const n = document.createElement('td'); n.textContent = r.note;
    tr.append(f, v, s, n); tb.appendChild(tr);
  });
}
function showAnalysis(j){
  sid = j.sid; $('ptype').value = j.part_type; $('tableCard').hidden = false; $('outCard').hidden = true;
  const conf = j.confidence != null ? ` (${Math.round(j.confidence*100)}% confident)` : '';
  const need = j.rows.filter(r => ['missing','not_in_text','llm_flag'].includes(r.status)).length;
  $('info').textContent = `Detected: ${j.part_type}${conf}. Read your text as: "${j.text_norm}". ` +
    (need ? `${need} value(s) need your attention.` : 'Everything was found - review, then confirm.');
  showRows(j.rows); say('');
}
function busy(b){ ['bExtract','bConfirm'].forEach(id => $(id).disabled = b); }

$('bExtract').onclick = async () => {
  busy(true);
  try { showAnalysis(await api('/api/analyze', {text: $('text').value})); }
  catch(e){ $('tableCard').hidden = false; say(e.message, 'err'); }
  busy(false);
};
$('ptype').onchange = async () => {
  if(!sid) return; busy(true);
  try { showAnalysis(await api('/api/retype', {sid, part_type: $('ptype').value})); } catch(e){ say(e.message, 'err'); }
  busy(false);
};
$('bConfirm').onclick = async () => {
  if(!sid) return; busy(true);
  const values = {}; document.querySelectorAll('#rows input').forEach(i => values[i.dataset.field] = i.value);
  try {
    const j = await api('/api/confirm', {sid, values});
    showRows(j.rows);
    const lines = j.issues.map(i => `- ${i.field}: ${i.msg}`).join('\n');
    if(!j.stl_url){ say("Can't build yet:\n" + lines, 'err'); $('outCard').hidden = true; }
    else {
      const s = j.info.size_mm.join(' x ');
      say(`Part generated: ${s} mm, volume ${j.info.volume_mm3} mm3, ${j.info.watertight ? 'watertight' : 'NOT watertight'}` + (lines ? '\nHeads-up:\n' + lines : ''), 'ok');
      $('outCard').hidden = false; $('json').textContent = JSON.stringify(j.params, null, 2);
      $('dl').href = j.stl_url + '?dl=1'; preview(j.stl_url);
    }
  } catch(e){ say(e.message, 'err'); }
  busy(false);
};
$('bReset').onclick = () => {
  sid = null; $('text').value = ''; $('tableCard').hidden = true; $('outCard').hidden = true; $('rows').textContent = ''; say('');
  if(renderer){ cancelAnimationFrame(raf); renderer.dispose(); renderer = null; }
  $('view').textContent = '3D preview';
};

function preview(url){
  const box = $('view'); box.textContent = 'loading preview...';
  if(renderer){ cancelAnimationFrame(raf); renderer.dispose(); renderer = null; }
  new STLLoader().load(url, geo => {
    box.textContent = '';
    const w = box.clientWidth, h = box.clientHeight;
    renderer = new THREE.WebGLRenderer({antialias:true}); renderer.setSize(w, h); box.appendChild(renderer.domElement);
    const scene = new THREE.Scene(); scene.background = new THREE.Color(0xeceff3);
    geo.computeVertexNormals(); geo.center(); geo.computeBoundingSphere();
    const R = geo.boundingSphere.radius;
    scene.add(new THREE.Mesh(geo, new THREE.MeshStandardMaterial({color:0x4a90d9, metalness:.1, roughness:.6})));
    scene.add(new THREE.AmbientLight(0xffffff, .7)); const d = new THREE.DirectionalLight(0xffffff, .9); d.position.set(1,2,3); scene.add(d);
    const cam = new THREE.PerspectiveCamera(40, w/h, R/100, R*100); cam.position.set(R*1.6, R*1.4, R*2.2);
    const ctl = new OrbitControls(cam, renderer.domElement);
    (function loop(){ raf = requestAnimationFrame(loop); ctl.update(); renderer.render(scene, cam); })();
  }, undefined, () => { box.textContent = '3D preview could not load (needs internet for three.js). The STL download still works.'; });
}
</script></body></html>"""

app = Flask("text_to_stl")
SESSIONS = {}

def _rows_json(rows):
    return [{"field": r["field"], "value": fmt_value(r["value"]), "status": r["status"],
             "label": STATUS_LABEL[r["status"]], "note": r["note"]} for r in rows]

def _analysis_json(sid, s):
    conf = max(s["probs"].values()) if s["probs"] else None
    return {"sid": sid, "part_type": s["part_type"], "confidence": conf, "text_norm": s["text_norm"], "rows": _rows_json(s["rows"])}

@app.get("/")
def index(): return INDEX_HTML

@app.post("/api/analyze")
def api_analyze():
    text = (request.get_json(silent=True) or {}).get("text", "").strip()
    if not text: return jsonify(error="Type a description of the part first."), 400
    try:
        s = analyze(text); sid = uuid.uuid4().hex; SESSIONS[sid] = s
        return jsonify(_analysis_json(sid, s))
    except Exception as e: return jsonify(error=f"{type(e).__name__}: {e}"), 500

@app.post("/api/retype")
def api_retype():
    b = request.get_json(silent=True) or {}
    old = SESSIONS.get(b.get("sid"))
    if old is None or b.get("part_type") not in PART_TYPES: return jsonify(error="Extract parameters first."), 400
    try:
        s = analyze(old["text"], forced_type=b["part_type"]); SESSIONS[b["sid"]] = s
        return jsonify(_analysis_json(b["sid"], s))
    except Exception as e: return jsonify(error=f"{type(e).__name__}: {e}"), 500

@app.post("/api/confirm")
def api_confirm():
    b = request.get_json(silent=True) or {}
    s = SESSIONS.get(b.get("sid"))
    if s is None: return jsonify(error="Extract parameters first."), 400
    try:
        res = confirm_and_build(s, {str(k): str(v) for k, v in (b.get("values") or {}).items()})
        s["rows"] = res["rows"]
        return jsonify({"rows": _rows_json(res["rows"]), "params": res["params"], "issues": res["issues"], "info": res["info"],
                        "stl_url": ("/stl/" + Path(res["stl_path"]).name) if res["stl_path"] else None})
    except Exception as e: return jsonify(error=f"{type(e).__name__}: {e}"), 500

@app.get("/stl/<name>")
def get_stl(name):
    if not re.fullmatch(r"[a-z_]+_\d+\.stl", name) or not (WORKDIR / name).exists(): abort(404)
    return send_file((WORKDIR / name).resolve(), mimetype="model/stl", as_attachment=request.args.get("dl") == "1", download_name=name)

# ---- start the server (re-running this cell restarts it cleanly)
PORT = 8765
if globals().get("_SERVER"): _SERVER.shutdown()
_SERVER = make_server("0.0.0.0", PORT, app, threaded=True)
threading.Thread(target=_SERVER.serve_forever, daemon=True).start()
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=950)      # shows the page right here in the notebook
    print("If the page doesn't show, use output.serve_kernel_port_as_window(PORT) for a separate tab.")
except ImportError:
    print(f"Running locally: open http://localhost:{PORT}")

## 11. Save the artifacts
Downloads a zip with the classifier, the LoRA adapter, the catalog tables, the split, and the feedback log — everything needed to move the app to a Hugging Face Space.

In [ ]:
# ------------------------------------------------------------------ SAVE everything you need to deploy on Hugging Face Spaces / for the report
ART = Path("artifacts"); ART.mkdir(exist_ok=True)
torch.save({"state_dict": clf.state_dict(), "vocab": VOCAB, "part_types": PART_TYPES}, ART / "classifier.pt")
model.save_pretrained(ART / "lora_adapter"); tok.save_pretrained(ART / "lora_adapter")
json.dump({"base_model": MODEL_NAME, "schema": SCHEMA, "standards": STANDARDS, "washer_lookup": WASHER_LOOKUP,
           "ranges": RANGES, "num_tol": NUM_TOL}, open(ART / "catalog_and_config.json", "w"), indent=1)
data[["sheet", "sheet_row", "split", "part_type", "text", "text_norm", "target"]].to_csv(ART / "dataset_with_split.csv", index=False)
if FEEDBACK_LOG.exists(): (ART / "feedback.jsonl").write_text(FEEDBACK_LOG.read_text())
with zipfile.ZipFile("artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ART.rglob("*"):
        if f.is_file(): z.write(f, f.relative_to(ART.parent))
print("saved:", sorted(p.name for p in ART.iterdir()))
try:
    from google.colab import files; files.download("artifacts.zip")
except ImportError:
    pass